# 23 · Patient timeline trên **nhãn thô 4 trạng thái** — U-Ones phóng đại bao nhiêu?

`16_patient_timeline.ipynb` (C16–C18) chạy trên `cohort_base.parquet`, tức nhãn **đã U-Ones hoá**.
Ở đó `-1` và `blank` đã bị nuốt mất, nên nhiều "thay đổi chẩn đoán" thực ra chỉ là **đổi trạng thái
nhắc tới** trong report chứ không phải bác sĩ đổi ý.

Ví dụ, dưới U-Ones (`+1/-1 → 1`, `0/blank → 0`):

| chuyển raw | U-Ones thấy | thực chất |
|---|---|---|
| `+1 → 0` | flip 1→0 | **bất đồng thật** — có bệnh vs không có bệnh |
| `+1 → blank` | flip 1→0 | report sau **không nhắc tới** nữa |
| `-1 → blank` | flip 1→0 | nghi ngờ, rồi thôi không nhắc |
| `-1 → 0` | flip 1→0 | nghi ngờ → loại trừ |
| `+1 → -1` | **không đổi** | chắc chắn → nghi ngờ |
| `0 → blank` | **không đổi** | loại trừ → không nhắc |

Notebook này tách bốn thứ đó ra, và vẽ lại timeline ảnh với nhãn thô.
**CPU thuần** (trừ phần đọc ảnh từ shard), ~2–4 phút.

## C0 · Config

In [ ]:
import os, io, re, json, tarfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from PIL import Image

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
LOCAL_IMG = os.environ.get('LOCAL_IMG', '/content/cxr_local')
DRIVE_IMG = '/content/drive/MyDrive/MIMIC_CXR/mimic-cxr-jpg-2.1.0.physionet.org/files'
SAVE_DIR  = os.path.join(DATA_DIR, 'timelines_raw')      # THU MUC MOI
os.makedirs(SAVE_DIR, exist_ok=True)
RAW_PQ = os.path.join(OUT_DIR, 'cohort_raw_labels.parquet')      # cache cua nb16, CHI DOC

SUBJECT_ID  = None        # None -> C4 tu chon theo PICK_RULE
SPLIT_FILTER = None
PICK_RULE   = 'most_uones_artifact'   # 'most_uones_artifact' | 'most_real_flip' | 'most_change'
MAX_STUDIES = 10
MAX_VIEWS   = 2
VIEW_KEEP   = ['frontal', 'lateral']
PREFER      = 'shards'
ROW_IN      = 3.0
SAVE_PNG    = True

# ---- ma trang thai tho (dung xuyen notebook) ----
BLANK, NEG, UNC, POS = 0, 1, 2, 3          # thu tu nay dung cho colormap
ST_NAME = {BLANK: 'blank', NEG: '0', UNC: '-1', POS: '+1'}
ST_COL  = {BLANK: '#e9ecef', NEG: '#3b82f6', UNC: '#f59e0b', POS: '#b91c1c'}
print('OUT_DIR =', OUT_DIR)
print('shards  =', 'present' if os.path.isdir(SHARD_DIR) else 'MISSING')

## C1 · Nhãn thô 4 trạng thái cho toàn cohort

`RAW` từ `cohort_raw_labels.parquet`; assert `U-Ones(raw)` tái tạo đúng nhãn `cohort_base`.
Nếu assert này fail thì mọi so sánh phía sau vô nghĩa.

In [ ]:
base = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_base.parquet'))
base['index_time'] = pd.to_datetime(base['index_time'])
META_COLS = {'study_id','subject_id','index_time','setting','cur_hadm','cur_edstay',
             'n_images','n_usable_view','n_frontal','n_lateral','n_view_ok','split'}
LABELS = [c for c in base.columns if c not in META_COLS]
C = len(LABELS); assert C == 13 and base.study_id.is_unique
NF = next(c for c in LABELS if c.replace('_', ' ').lower() == 'no finding')
PATH = [c for c in LABELS if c != NF]
vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))

rawdf = pd.read_parquet(RAW_PQ)
RAWM = (rawdf.drop_duplicates('study_id').set_index('study_id')
        .reindex(base.study_id.to_numpy())[[f'raw_{l}' for l in LABELS]].to_numpy('float64'))
UONES = base[LABELS].to_numpy('int8')
_chk = np.where(np.isnan(RAWM), 0, np.where(RAWM == -1, 1, RAWM)).astype('int8')
_bad = int((_chk != UONES).sum())
print(f'U-Ones(raw) == cohort_base ? {_bad == 0}  ({_bad} o lech)')
assert _bad == 0, 'nhan tho khong tai tao duoc nhan cohort -> dung'

# ---- ma hoa 4 trang thai ----
ST = np.full((len(base), C), BLANK, 'int8')
ST[RAWM == 0] = NEG; ST[RAWM == -1] = UNC; ST[RAWM == 1] = POS
base_st = pd.DataFrame(ST, columns=LABELS)
print(f'studies {len(base):,} | subjects {base.subject_id.nunique():,} | labels {C}')
print('phan bo trang thai (toan cohort):',
      {ST_NAME[k]: int((ST == k).sum()) for k in (POS, NEG, UNC, BLANK)})

## C2 · Cặp study liên tiếp — **U-Ones thấy bao nhiêu thay đổi, thật ra bao nhiêu?**

Tái lập đúng C16 của `16_patient_timeline` (cột `pct_changed` dùng U-Ones), rồi thêm ba định nghĩa
chặt hơn tính trên nhãn thô:

```
uones_flip     y=(raw in {+1,-1}) doi giua hai study      <- dinh nghia cu
real_flip      +1 <-> 0            bat dong THAT
certainty      -1 <-> +1 / -1 <-> 0   doi do chac chan, cung mot huong benh
mention        blank <-> bat ky      xuat hien/bien mat khoi report
```

In [ ]:
d = base.sort_values(['subject_id', 'index_time'], kind='stable').reset_index(drop=True)
Sd = base_st.loc[d.index if False else base.sort_values(
    ['subject_id', 'index_time'], kind='stable').index].to_numpy('int8')
sid = d.subject_id.to_numpy(); sids = d.study_id.to_numpy()
t = d.index_time.to_numpy('datetime64[s]').astype('int64')
same = sid[1:] == sid[:-1]
A, B = Sd[:-1][same], Sd[1:][same]                      # trang thai tho truoc / sau
gap_h = (t[1:] - t[:-1])[same] / 3600.0
assert (gap_h >= -1e-9).all(), 'gap am -> sort sai'

U = lambda s: (s == POS) | (s == UNC)                    # U-Ones
uflip  = U(A) != U(B)
real   = ((A == POS) & (B == NEG)) | ((A == NEG) & (B == POS))
certa  = (((A == UNC) & ((B == POS) | (B == NEG))) | ((B == UNC) & ((A == POS) | (A == NEG))))
mention = ((A == BLANK) & (B != BLANK)) | ((B == BLANK) & (A != BLANK))
nochg  = A == B

GAP_EDGES = [0, 1, 6, 24, 24*7, 24*365*100]
GAP_NAMES = ['<1h', '1-6h', '6-24h', '1-7d', '>7d']
bucket = pd.cut(gap_h, bins=GAP_EDGES, labels=GAP_NAMES, right=False)
P = pd.DataFrame(dict(subject_id=sid[1:][same], prev_study=sids[:-1][same],
                      study_id=sids[1:][same], gap_h=gap_h, bucket=bucket,
                      uones_changed=uflip.any(1), real_changed=real.any(1),
                      certainty_changed=certa.any(1), mention_changed=mention.any(1),
                      n_uflip=uflip.sum(1), n_real=real.sum(1),
                      n_certainty=certa.sum(1), n_mention=mention.sum(1)))
P.to_parquet(f'{SAVE_DIR}/pairs_raw_state.parquet', index=False)
print(f'consecutive pairs: {len(P):,}\n')

g = P.groupby('bucket', observed=False)
tab = pd.DataFrame({'n_pairs': g.size(),
                    'pct_of_all': (g.size()/len(P)*100).round(2),
                    'U-ONES %chg': (g.uones_changed.mean()*100).round(1),
                    'REAL(+1<->0) %chg': (g.real_changed.mean()*100).round(1),
                    'certainty %chg': (g.certainty_changed.mean()*100).round(1),
                    'mention %chg': (g.mention_changed.mean()*100).round(1),
                    'median_gap_h': g.gap_h.median().round(2)})
print(tab.to_string())
print(f'\ntoan bo cap: U-Ones {P.uones_changed.mean():.1%} | '
      f'REAL {P.real_changed.mean():.1%} | '
      f'certainty {P.certainty_changed.mean():.1%} | mention {P.mention_changed.mean():.1%}')

# ---- U-Ones flip den TU DAU? phan ra theo chuyen trang thai ----
pairs_state = (A.astype('int16') * 4 + B.astype('int16'))[uflip]
lab = [f'{ST_NAME[a]} -> {ST_NAME[b]}' for a in range(4) for b in range(4)]
cnt = np.bincount(pairs_state.ravel(), minlength=16)
FL = pd.DataFrame(dict(chuyen=lab, n_cell=cnt))
FL = FL[FL.n_cell > 0].sort_values('n_cell', ascending=False)
FL['pct_cua_uones_flip'] = (FL.n_cell/FL.n_cell.sum()*100).round(2)
FL['la_bat_dong_that'] = FL.chuyen.isin(['+1 -> 0', '0 -> +1'])
print(f'\n=== {int(uflip.sum()):,} CELL bi U-Ones tinh la FLIP den tu dau? ===')
print(FL.to_string(index=False))
_real = int(FL[FL.la_bat_dong_that].n_cell.sum()); _tot = int(FL.n_cell.sum())
print(f'\n  bat dong THAT (+1<->0) : {_real:>9,} / {_tot:,} = {_real/_tot:.1%} so flip cua U-Ones')
print(f'  do blank / -1          : {_tot-_real:>9,} / {_tot:,} = {1-_real/_tot:.1%}')
print(f'  -> U-Ones phong dai so thay doi len {_tot/max(_real,1):.1f} lan')

## C3 · Per-label — nhãn nào thật sự bất đồng, nhãn nào chỉ đổi trạng thái nhắc tới

Tái lập C18 của `16_patient_timeline` (`unstable_%` trên cặp `<6h`, U-Ones) rồi đặt cạnh
`unstable_real_%` chỉ đếm `+1 ↔ 0`.

In [ ]:
M6 = (P.gap_h < 6).to_numpy()
A6, B6 = A[M6], B[M6]
u6 = (U(A6) != U(B6))
either_u = (U(A6) | U(B6))
r6 = ((A6 == POS) & (B6 == NEG)) | ((A6 == NEG) & (B6 == POS))
either_r = ((A6 == POS) | (A6 == NEG)) & ((B6 == POS) | (B6 == NEG))   # ca hai deu duoc nhac VA dut khoat

per = pd.DataFrame({
    'prevalence_%': (UONES.mean(0)*100).round(2),
    'n_pairs_uones_pos': either_u.sum(0),
    'unstable_uones_%': np.where(either_u.sum(0) > 0,
                                 u6.sum(0)/np.maximum(either_u.sum(0), 1)*100, np.nan).round(1),
    'n_pairs_both_definite': either_r.sum(0),
    'unstable_real_%': np.where(either_r.sum(0) > 0,
                                r6.sum(0)/np.maximum(either_r.sum(0), 1)*100, np.nan).round(1),
}, index=LABELS).sort_values('prevalence_%')
per['n_mention_only'] = (((A6 == BLANK) != (B6 == BLANK)).sum(0))
print(f'pairs <6h dung: {int(M6.sum()):,}\n')
print(per.to_string())
print('''
  unstable_uones_%      : mau so = cap co it nhat mot ben U-Ones=1  (dinh nghia cu, C18 nb16)
  unstable_real_%       : mau so = cap ma CA HAI ben deu duoc nhac VA dut khoat (+1 hoac 0)
                          -> chi con bat dong lam sang that su
  n_mention_only        : so cap ma mot ben blank, ben kia co nhac -> khong phai bat dong''')

# ---- collapse / flicker tren nhan tho ----
jNF = LABELS.index(NF)
prev_pos_real = (A[:, [i for i in range(C) if i != jNF]] == POS).sum(1)
cur_pos_real  = (B[:, [i for i in range(C) if i != jNF]] == POS).sum(1)
cur_nf = B[:, jNF] == POS
col_u = (U(A[:, [i for i in range(C) if i != jNF]]).sum(1) >= 1) & cur_nf & \
        (U(B[:, [i for i in range(C) if i != jNF]]).sum(1) == 0)
col_r = (prev_pos_real >= 1) & cur_nf & (cur_pos_real == 0)
print(f'\n=== COLLAPSE: co benh -> No Finding sach ===')
print(f'  dinh nghia U-Ones : {int(col_u.sum()):>7,} cap ({col_u.mean():.2%})  <- so cua nb16')
print(f'  dinh nghia raw +1 : {int(col_r.sum()):>7,} cap ({col_r.mean():.2%})')
print(f'  <6h: U-Ones {int((col_u & M6).sum()):,} | raw {int((col_r & M6).sum()):,}')

## C4 · Chọn bệnh nhân để vẽ

In [ ]:
n_st = d.groupby('subject_id').size()
cand = set(n_st[(n_st >= 4) & (n_st <= MAX_STUDIES)].index)
if SPLIT_FILTER is not None:
    cand &= set(d[d.split == SPLIT_FILTER].subject_id)
Pc = P[P.subject_id.isin(cand)]
if SUBJECT_ID is not None:
    SUBJ = int(SUBJECT_ID)
else:
    if PICK_RULE == 'most_uones_artifact':
        sc = (Pc.n_uflip - Pc.n_real).groupby(Pc.subject_id).sum()
    elif PICK_RULE == 'most_real_flip':
        sc = Pc.n_real.groupby(Pc.subject_id).sum()
    else:
        sc = Pc.n_uflip.groupby(Pc.subject_id).sum()
    assert len(sc), 'khong co subject nao thoa dieu kien'
    SUBJ = int(sc.sort_values(ascending=False, kind='stable').index[0])
    print(f'auto-picked subject_id = {SUBJ}  (PICK_RULE={PICK_RULE})')

st = (base[base.subject_id == SUBJ].sort_values('index_time', kind='stable').reset_index(drop=True))
STs = base_st.loc[base[base.subject_id == SUBJ].sort_values('index_time', kind='stable').index
                  ].to_numpy('int8')
if len(st) > MAX_STUDIES:
    print(f'!! subject co {len(st)} study; ve {MAX_STUDIES} cai DAU (cu nhat)')
    st = st.head(MAX_STUDIES).copy(); STs = STs[:MAX_STUDIES]
t0 = st.index_time.iloc[0]
st['day'] = (st.index_time - t0).dt.total_seconds()/86400.0
st['gap_h'] = st.index_time.diff().dt.total_seconds()/3600.0

imgs = (vm[vm.study_id.isin(set(st.study_id)) & vm.view_group.isin(VIEW_KEEP)]
        .merge(st[['study_id', 'index_time']], on='study_id')
        .sort_values(['index_time', 'view_group', 'dicom_id'], kind='stable'))
print(f'\nsubject {SUBJ}: {len(st)} study, {len(imgs)} anh | '
      f'{t0.date()} -> {st.index_time.iloc[-1].date()}')

rows = []
for i in range(len(st)):
    r = st.iloc[i]
    cell = {l: ST_NAME[STs[i, j]] for j, l in enumerate(LABELS)}
    rows.append(dict(n=i+1, study_id=r.study_id, when=f'{r.index_time:%Y-%m-%d %H:%M}',
                     gap_h=('-' if i == 0 else f'{r.gap_h:.1f}'), setting=r.setting, **cell))
TL = pd.DataFrame(rows)
print()
print(TL.to_string(index=False))

print('\n=== DOI GIUA HAI STUDY LIEN TIEP (tho) ===')
for i in range(1, len(st)):
    ch = []
    for j, l in enumerate(LABELS):
        a, b = STs[i-1, j], STs[i, j]
        if a == b: continue
        kind = ('BAT DONG THAT' if {a, b} == {POS, NEG} else
                'mention' if BLANK in (a, b) else 'do chac chan')
        ch.append(f'{l}: {ST_NAME[a]}->{ST_NAME[b]} [{kind}]')
    print(f'  #{i} -> #{i+1}  (+{st.gap_h.iloc[i]:.1f}h)  ' + ('; '.join(ch) if ch else '(khong doi)'))

## C5 · Image loader — shards trước, originals dự phòng *(verbatim nb16 C4)*

In [ ]:
_wl_path = os.path.join(SHARD_DIR, '_work_list.parquet')
_wl = (pd.read_parquet(_wl_path)[['dicom_id', 'shard']].set_index('dicom_id')['shard']
       if os.path.exists(_wl_path) else pd.Series(dtype='int64'))
_tar_cache = {}

def _from_local(dic):
    p = os.path.join(LOCAL_IMG, f'{dic}.jpg')
    return (open(p, 'rb').read(), 'local') if os.path.exists(p) else (None, None)

def _from_shard(dic):
    if dic not in _wl.index: return None, None
    i = int(_wl.loc[dic]); p = os.path.join(SHARD_DIR, f'shard_{i:05d}.tar')
    if not os.path.exists(p): return None, None
    t_ = _tar_cache.get(i) or _tar_cache.setdefault(i, tarfile.open(p))
    for name in (f'{dic}.jpg', f'./{dic}.jpg'):
        try: return t_.extractfile(t_.getmember(name)).read(), f'shard{i}'
        except KeyError: continue
    return None, None

def _from_original(dic, sub, stu):
    s = str(int(sub)); p = os.path.join(DRIVE_IMG, f'p{s[:2]}/p{s}/s{int(stu)}/{dic}.jpg')
    return (open(p, 'rb').read(), 'orig') if os.path.exists(p) else (None, None)

def load_img(dic, sub, stu):
    order = ([_from_local, _from_shard, _from_original] if PREFER == 'shards'
             else [_from_original, _from_local, _from_shard])
    for fn in order:
        raw, src = (fn(dic, sub, stu) if fn is _from_original else fn(dic))
        if raw is not None:
            return Image.open(io.BytesIO(raw)).convert('L'), src
    return None, 'MISSING'

pix = {}
for _, r in imgs.iterrows():
    pix[r.dicom_id] = load_img(r.dicom_id, SUBJ, r.study_id)
n_miss = sum(1 for v in pix.values() if v[0] is None)
print(f'loaded {len(pix)-n_miss}/{len(pix)}' + (f'  !! {n_miss} missing' if n_miss else ''))

## C6 · Hình A — timeline ảnh, chú thích bằng **nhãn thô**

Mỗi hàng một study. Cột nhãn có **axes riêng** (`gridspec`) nên chữ không bao giờ đè lên ảnh.
Mỗi nhãn một ô màu theo trạng thái thô: 🟥 `+1` · 🟧 `-1` · 🟦 `0` · ⬜ `blank`.
**Viền đen** quanh ô = trạng thái đổi so với study liền trước.

Núm chỉnh:

* `SHOW_ONLY_CHANGED = True` — chỉ vẽ nhãn **có đổi ít nhất một lần** trong dải study đang xem.
  Đặt `False` để xem đủ 13 nhãn (chỉ bỏ nhãn `blank` ở mọi study).
* `LABEL_COL_W = 1.15` — bề rộng cột nhãn, đơn vị = một ảnh.
* `ROW_IN` (C0) — chiều cao mỗi hàng; `MAX_VIEWS = 1` cho hình gọn hơn nhiều nếu đa số study
  chỉ có một phim frontal.

In [ ]:
SHOW_ONLY_CHANGED = True     # True: chi ve nhan CO doi it nhat mot lan (bot nhieu han)
LABEL_COL_W       = 1.15     # be rong cot nhan, don vi = mot anh

by_study = {k: v.head(MAX_VIEWS) for k, v in imgs.groupby('study_id')}
ncol = max((len(v) for v in by_study.values()), default=1)
nrow = len(st)
if SHOW_ONLY_CHANGED:
    keep = [j for j in range(C) if len(set(STs[:, j].tolist())) > 1]
    if not keep: keep = [j for j in range(C) if not (STs[:, j] == BLANK).all()]
    print(f've {len(keep)}/{C} nhan CO thay doi (dat SHOW_ONLY_CHANGED=False de xem het)')
else:
    keep = [j for j in range(C) if not (STs[:, j] == BLANK).all()]
    print(f've {len(keep)}/{C} nhan (bo {C-len(keep)} nhan blank o MOI study)')

# Cot nhan co AXES RIENG -> khong the de len anh.
fig = plt.figure(figsize=(ROW_IN*(ncol + LABEL_COL_W) + 0.5, ROW_IN*nrow))
gs = fig.add_gridspec(nrow, ncol + 1, width_ratios=[LABEL_COL_W] + [1]*ncol,
                      wspace=0.03, hspace=0.12)
fig.suptitle(f'subject {SUBJ} · {nrow} study, cu -> moi · NHAN THO 4 trang thai\n'
             f'vien den quanh o = doi so voi study lien truoc',
             fontsize=13, y=0.998)
Y_TOP = 0.835                                    # chip dau tien, DUOI dong timestamp
dy = min(0.072, (Y_TOP - 0.02)/max(len(keep), 1))
for i in range(nrow):
    r = st.iloc[i]
    axL = fig.add_subplot(gs[i, 0]); axL.axis('off')
    axL.text(0, 1.0, f'#{i+1}   ' + ('baseline' if i == 0 else f'+{r.gap_h:.1f}h'),
             transform=axL.transAxes, fontsize=11, fontweight='bold', va='top', ha='left')
    axL.text(0, 1.0 - 0.062, f'{r.index_time:%Y-%m-%d %H:%M}  ·  {r.setting}  ·  s{r.study_id}',
             transform=axL.transAxes, fontsize=8, color='#4b5563', va='top', ha='left')
    for k, j in enumerate(keep):
        s = STs[i, j]
        chg = i > 0 and STs[i-1, j] != s
        axL.text(0.015, Y_TOP - k*dy, f'{LABELS[j]}  {ST_NAME[s]}',
                 transform=axL.transAxes, fontsize=8.5, va='top', ha='left',
                 color='white' if s != BLANK else '#6b7280',
                 bbox=dict(boxstyle='round,pad=0.28', fc=ST_COL[s],
                           ec='black' if chg else 'none', lw=1.6))
    rws = by_study.get(r.study_id, imgs.iloc[:0])
    for j2 in range(ncol):
        ax = fig.add_subplot(gs[i, j2 + 1]); ax.set_xticks([]); ax.set_yticks([])
        if j2 >= len(rws): ax.axis('off'); continue
        g = rws.iloc[j2]; im, src = pix[g.dicom_id]
        if im is None:
            ax.text(.5, .5, 'image not found', ha='center', va='center', color='crimson')
            ax.axis('off'); continue
        ax.imshow(im, cmap='gray')
        ax.set_title(f'{g.view_group} · {src}', fontsize=9, pad=3)
        for sp in ax.spines.values(): sp.set_color('#d1d5db')
if SAVE_PNG:
    out = f'{SAVE_DIR}/timeline_raw_subject_{SUBJ}.png'
    fig.savefig(out, dpi=110, bbox_inches='tight'); print('saved ->', out)
plt.show()

## C7 · Hình B — **RAW vs U-ONES cạnh nhau**

Trái: trạng thái thô 4 màu. Phải: cùng dữ liệu sau U-Ones (chỉ còn 2 màu).

Flip là thuộc tính của **chuyển tiếp giữa hai study**, không phải của một ô — nên nó được vẽ thành
**vạch dọc nằm trên đường biên giữa hai cột**, ở cả hai bảng:

* ⬛ vạch đen mảnh — U-Ones **tạo ra** một flip mà nhãn thô không hề mâu thuẫn
  (`+1↔blank`, `-1↔blank`, `-1↔0`)
* 🟩 vạch xanh dày — **bất đồng thật** `+1 ↔ 0`

Đọc bảng trái ở hai cột kề vạch để thấy chuyển tiếp thực sự là gì.

In [ ]:
UO = ((STs == POS) | (STs == UNC)).astype('int8')
cmap_raw = ListedColormap([ST_COL[BLANK], ST_COL[NEG], ST_COL[UNC], ST_COL[POS]])
cmap_uo  = ListedColormap(['#e9ecef', '#b91c1c'])
xt = [f'#{i+1}\n' + ('base' if i == 0 else f'+{st.gap_h.iloc[i]:.0f}h') for i in range(nrow)]

fig, ax = plt.subplots(1, 2, figsize=(1.15*nrow + 7.5, 0.46*C + 2.4),
                       gridspec_kw=dict(width_ratios=[1, 1]))
ax[0].imshow(STs[:, :].T, cmap=cmap_raw, vmin=-0.5, vmax=3.5, aspect='auto')
ax[0].set_title('RAW — 4 trang thai', fontsize=12)
ax[1].imshow(UO.T, cmap=cmap_uo, vmin=-0.5, vmax=1.5, aspect='auto')
ax[1].set_title('U-ONES — +1/-1 -> 1, 0/blank -> 0', fontsize=12)
for a in ax:
    a.set_xticks(range(nrow)); a.set_xticklabels(xt, fontsize=8)
    a.set_yticks(range(C)); a.set_yticklabels(LABELS, fontsize=8)
    a.set_xticks(np.arange(-.5, nrow, 1), minor=True)
    a.set_yticks(np.arange(-.5, C, 1), minor=True)
    a.grid(which='minor', color='white', lw=1.2); a.tick_params(which='minor', length=0)
C_ART, C_REAL = '#111827', '#16a34a'
# Flip la thuoc tinh cua CHUYEN TIEP giua hai study, khong phai cua mot o.
# -> ve VACH DOC nam tren duong bien giua cot i-1 va cot i, o CA HAI bang.
for i in range(1, nrow):
    for j in range(C):
        a_, b_ = STs[i-1, j], STs[i, j]
        if ((a_ in (POS, UNC)) != (b_ in (POS, UNC))):
            _is_real = {a_, b_} == {POS, NEG}      # ten rieng: KHONG de len mang `real` cua C2
            for a_x in ax:
                a_x.plot([i-0.5, i-0.5], [j-0.44, j+0.44], solid_capstyle='butt', zorder=5,
                         color=C_REAL if _is_real else C_ART, lw=5 if _is_real else 2.6)
for k, v in ST_NAME.items():
    ax[0].plot([], [], 's', c=ST_COL[k], label=v, ms=9, ls='none')
ax[0].legend(loc='upper left', bbox_to_anchor=(0, -0.10), ncol=4, fontsize=9, frameon=False,
             title='trang thai o', title_fontsize=9)
ax[1].plot([], [], color=C_ART, lw=2.6, label='U-Ones TAO flip (do blank / -1)')
ax[1].plot([], [], color=C_REAL, lw=5, label='bat dong THAT  +1 <-> 0')
ax[1].legend(loc='upper left', bbox_to_anchor=(0, -0.10), ncol=2, fontsize=9, frameon=False,
             title='vach doc = thay doi GIUA hai study lien ke', title_fontsize=9)
fig.suptitle(f'subject {SUBJ} — nhan tho vs U-Ones', fontsize=13)
plt.tight_layout(rect=[0, 0.02, 1, 0.96])
if SAVE_PNG:
    out = f'{SAVE_DIR}/raw_vs_uones_subject_{SUBJ}.png'
    fig.savefig(out, dpi=120, bbox_inches='tight'); print('saved ->', out)
plt.show()

nflip = int(sum(((STs[i-1, j] in (POS, UNC)) != (STs[i, j] in (POS, UNC)))
                for i in range(1, nrow) for j in range(C)))
nreal = int(sum(({STs[i-1, j], STs[i, j]} == {POS, NEG})
                for i in range(1, nrow) for j in range(C)))
print(f'\nsubject {SUBJ}: U-Ones tao {nflip} flip, trong do {nreal} la bat dong that '
      f'(+1<->0) -> {nflip-nreal} flip la do blank/-1')